# Milestone M5: Eksekusi Baseline Komparatif (TF-IDF vs Machine Learning Tradisional)
## Kelompok 4 — CNN for Text Classification (Indonesian Hate Speech Detection)

**Mata Kuliah:** Workshop Proyek Sistem Cerdas 2026  
**Dosen Pengampu:** Dr. Selvia Ferdiana Kusuma, M.Kom  
**Dataset:** `data/splits/train.csv` (~19.900 baris) & `data/splits/test.csv` (~4.260 baris)  

### Tujuan & Agenda Eksperimen:
1. **Pemuatan Data Partisi Asli**: Membaca dataset train & test yang diproduksi pada M3–M4.
2. **Ekstraksi Fitur TF-IDF**: Unigram + Bigram ($1, 2$) dengan batasan $10.000$ fitur teratas.
3. **Training & Evaluasi Model Baseline Komparatif**:
   - **Logistic Regression** (Balanced Class Weights)
   - **Linear Support Vector Classifier (LinearSVC)**
   - **Multinomial Naive Bayes**
4. **Kalkulasi Metrik Lengkap (`MetricCalculator`)**: Macro-F1 (Metrik Utama), Macro-Precision, Macro-Recall, Per-Class F1, Confusion Matrix.
5. **Penyimpanan Laporan Metrik**: Serialisasi metrik ke `outputs/metrics/baseline_tfidf.json` sebagai baseline acuan untuk model CNN M6–M7.

In [2]:
import sys
from pathlib import Path
import os

# Set root path project
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from src.utils.config import Config
from src.utils.seed import set_seed
from src.evaluation.metrics import MetricCalculator
from src.evaluation.confusion import ConfusionMatrixPlotter

set_seed(Config.SEED)
print(f"Project root: {ROOT_DIR}")
print(f"Random seed : {Config.SEED}")

Project root: /home/josjiez/Documents/IndoToxic
Random seed : 42


### 1. Pemuatan Data Partisi Split Asli (`data/splits/`)

In [3]:
train_csv = ROOT_DIR / Config.TRAIN_CSV
val_csv = ROOT_DIR / Config.VAL_CSV
test_csv = ROOT_DIR / Config.TEST_CSV

if not os.path.exists(train_csv) or not os.path.exists(test_csv):
    raise FileNotFoundError(f"File split belum dibuat. Harap jalankan 02_preprocessing.ipynb terlebih dahulu.")

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

print(f"Train Data : {len(train_df):,} baris | Rasio Toxic: {train_df['label'].mean()*100:.2f}%")
print(f"Val Data   : {len(val_df):,} baris | Rasio Toxic: {val_df['label'].mean()*100:.2f}%")
print(f"Test Data  : {len(test_df):,} baris | Rasio Toxic: {test_df['label'].mean()*100:.2f}%")

X_train_text = train_df["text_clean"].astype(str).values
y_train = train_df["label"].values

X_test_text = test_df["text_clean"].astype(str).values
y_test = test_df["label"].values

Train Data : 19,911 baris | Rasio Toxic: 14.04%
Val Data   : 4,267 baris | Rasio Toxic: 14.06%
Test Data  : 4,267 baris | Rasio Toxic: 14.04%


### 2. Ekstraksi Fitur TF-IDF (N-gram 1-2, Max Features = 10.000)

In [4]:
print("Mengekstraksi fitur TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# Fit HANYA pada train corpus
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print(f"Bentuk Matrix TF-IDF Train: {X_train_tfidf.shape}")
print(f"Bentuk Matrix TF-IDF Test : {X_test_tfidf.shape}")

Mengekstraksi fitur TF-IDF...
Bentuk Matrix TF-IDF Train: (19911, 10000)
Bentuk Matrix TF-IDF Test : (4267, 10000)


### 3. Model 1: Logistic Regression (Balanced Class Weight)

In [5]:
lr_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=Config.SEED)
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

metric_calculator = MetricCalculator()
metrics_lr = metric_calculator.compute_all(y_test, y_pred_lr)

print("\n=======================================================")
print("  HASIL EVALUASI BASELINE: LOGISTIC REGRESSION (TF-IDF)")
print("=======================================================")
print(f"Macro-F1 Score   : {metrics_lr['macro_f1']:.4f} (Primary Metric)")
print(f"Macro-Precision  : {metrics_lr['precision_macro']:.4f}")
print(f"Macro-Recall     : {metrics_lr['recall_macro']:.4f}")
print(f"Accuracy         : {metrics_lr['accuracy']:.4f}")
print(f"Non-toxic F1     : {metrics_lr['per_class']['non_toxic']['f1']:.4f}")
print(f"Toxic F1         : {metrics_lr['per_class']['toxic']['f1']:.4f}")


  HASIL EVALUASI BASELINE: LOGISTIC REGRESSION (TF-IDF)
Macro-F1 Score   : 0.6574 (Primary Metric)
Macro-Precision  : 0.6400
Macro-Recall     : 0.7186
Accuracy         : 0.7839
Non-toxic F1     : 0.8656
Toxic F1         : 0.4492


### 4. Model 2 & 3: Linear SVM & Naive Bayes Baseline

In [6]:
# Linear SVM
svm_model = LinearSVC(class_weight="balanced", random_state=Config.SEED, max_iter=2000)
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)
metrics_svm = metric_calculator.compute_all(y_test, y_pred_svm)

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)
metrics_nb = metric_calculator.compute_all(y_test, y_pred_nb)

benchmark_df = pd.DataFrame([
    {"Model": "Logistic Regression (Balanced)", "Macro-F1": metrics_lr["macro_f1"], "Toxic-Recall": metrics_lr["per_class"]["toxic"]["recall"], "Accuracy": metrics_lr["accuracy"]},
    {"Model": "Linear SVM (Balanced)", "Macro-F1": metrics_svm["macro_f1"], "Toxic-Recall": metrics_svm["per_class"]["toxic"]["recall"], "Accuracy": metrics_svm["accuracy"]},
    {"Model": "Multinomial Naive Bayes", "Macro-F1": metrics_nb["macro_f1"], "Toxic-Recall": metrics_nb["per_class"]["toxic"]["recall"], "Accuracy": metrics_nb["accuracy"]}
]).sort_values(by="Macro-F1", ascending=False)

print("Tabel Rekapitulasi Baseline ML:")
print(benchmark_df.to_string(index=False))

Tabel Rekapitulasi Baseline ML:
                         Model  Macro-F1  Toxic-Recall  Accuracy
Logistic Regression (Balanced)  0.657411      0.627713  0.783923
         Linear SVM (Balanced)  0.642235      0.537563  0.786970
       Multinomial Naive Bayes  0.549873      0.105175  0.861261


### 5. Visualisasi Confusion Matrix & Ekspor Artefak JSON

In [7]:
cm_plotter = ConfusionMatrixPlotter(class_names=["Non-toxic", "Toxic"])
cm_save_path = ROOT_DIR / "outputs/plots/confusion_matrix_baseline_tfidf.png"
cm_plotter.plot_and_save(y_test, y_pred_lr, output_path=str(cm_save_path), title="Confusion Matrix: TF-IDF + Logistic Regression (Baseline)")
print(f"Confusion matrix plot tersimpan di: {cm_save_path}")

# Simpan metrik baseline ke outputs/metrics/
metric_out_path = ROOT_DIR / "outputs/metrics/baseline_tfidf.json"
metric_calculator.save_metrics(metrics_lr, str(metric_out_path))
print(f"Laporan metrik baseline tersimpan di: {metric_out_path}")

Confusion matrix plot tersimpan di: /home/josjiez/Documents/IndoToxic/outputs/plots/confusion_matrix_baseline_tfidf.png
Laporan metrik baseline tersimpan di: /home/josjiez/Documents/IndoToxic/outputs/metrics/baseline_tfidf.json


### 6. Kesimpulan Milestone M5
1. **Ambang Batas Minimum**: Baseline `TF-IDF + Logistic Regression (Balanced)` menghasilkan Macro-F1 acuan untuk korpus `indotoxic2024`.
2. **Tantangan Imbalance**: Model baseline tanpa penyeimbang bobot cenderung gagal mendeteksi kelas toxic minoritas (Recall rendah).
3. **Langkah M6–M7**: Arsitektur Deep Learning **CNN Multi-kernel** akan diuji untuk melampaui skor Macro-F1 baseline ini.